In [1]:
from data_collection.data_adapters import DatabentoGenericAdapter
from dotenv import load_dotenv
from pathlib import Path
import os
import polars as pl
import numpy as np
import matplotlib.pyplot as plt
load_dotenv()
pl.enable_string_cache()

CACHE_DIR = Path.cwd().resolve().parents[0] / "data" / "dbn_cache" / "generic"
adapter = DatabentoGenericAdapter(os.getenv("DATABENTO-API-KEY"), CACHE_DIR)

print(f"CACHE_DIR: {CACHE_DIR}")
# Clear the cache if needed
# adapter.clear_cache()

with open("../universe.txt", "r") as f:
    UNIVERSE = [line.strip() for line in f]
START = "2023-06-01"
END = "2025-06-01"
DATASET = "XNAS.ITCH"

df_bars = (
    adapter.scan_data(
        tickers="ALL_SYMBOLS",
        start_date=START,
        end_date=END,
        dataset=DATASET,
        schema="ohlcv-1d"
    )
    .select(["ts_event", "symbol", "close"])
    .with_columns(
        pl.col("symbol").cast(pl.Categorical)
    )
    .filter(
        pl.col("symbol").is_in(UNIVERSE)
    )
    .sort(["ts_event", "symbol"])
)

CACHE_DIR: /Users/michaelcusack-nelkin/GitHub/Summer_Research_2026/data/dbn_cache/generic
[GenericAdapter] scan_parquet: 548cfb7cc98703d7d8a8a14fe79dfe1a.parquet


In [2]:
CAX = (
     pl.read_csv("../CAX/corporate_actions.csv")
     .sort(["date", "symbol"])
     .with_columns(
         pl.col("date").str.to_datetime(time_unit="ns", time_zone="UTC"),
         pl.col("symbol").cast(pl.Categorical)
     )
)
CAX

symbol,date,dividend,split
cat,"datetime[ns, UTC]",f64,f64
"""AGG""",2023-06-01 00:00:00 UTC,0.258,0.0
"""AJG""",2023-06-01 00:00:00 UTC,0.55,0.0
"""ALL""",2023-06-01 00:00:00 UTC,0.89,0.0
"""ANGL""",2023-06-01 00:00:00 UTC,0.131,0.0
"""BAC""",2023-06-01 00:00:00 UTC,0.22,0.0
…,…,…,…
"""FIS""",2026-06-11 00:00:00 UTC,0.44,0.0
"""FNV""",2026-06-11 00:00:00 UTC,0.44,0.0
"""JXN""",2026-06-11 00:00:00 UTC,0.9,0.0


In [3]:
daily_px = (
      df_bars.collect(engine="streaming")
      .join(CAX, left_on=["ts_event", "symbol"], right_on=["date", "symbol"], how="left")
      .sort(["symbol", "ts_event"])
      .with_columns(
          pl.col("dividend").fill_null(0.0),
          # 0.0 / null == "no split" -> ratio of 1.0, not 0.0
          pl.when(pl.col("split").fill_null(0.0) == 0.0)
            .then(1.0).otherwise(pl.col("split")).alias("split"),
      )
      .with_columns(
          (pl.col("split") / (1 - pl.col("dividend") / pl.col("close").shift().over("symbol"))).alias("CAX")
      )
      .with_columns(pl.col("CAX").fill_null(1.0))            # first row per symbol: no prev_close
      .with_columns(pl.col("CAX").cum_prod().over("symbol"))
  )
daily_px

ts_event,symbol,close,dividend,split,CAX
"datetime[ns, UTC]",cat,f64,f64,f64,f64
2023-06-01 00:00:00 UTC,"""A""",116.0,0.0,1.0,1.0
2023-06-02 00:00:00 UTC,"""A""",118.5,0.0,1.0,1.0
2023-06-05 00:00:00 UTC,"""A""",118.88,0.0,1.0,1.0
2023-06-06 00:00:00 UTC,"""A""",116.83,0.0,1.0,1.0
2023-06-07 00:00:00 UTC,"""A""",116.0,0.0,1.0,1.0
…,…,…,…,…,…
2025-05-23 00:00:00 UTC,"""ZWS""",35.53,0.0,1.0,1.020103
2025-05-27 00:00:00 UTC,"""ZWS""",36.92,0.0,1.0,1.020103
2025-05-28 00:00:00 UTC,"""ZWS""",36.23,0.0,1.0,1.020103


In [4]:
adj_px = (
    daily_px
    .with_columns((pl.col("close") * pl.col("CAX")).alias("adj_close"))
    .select(["ts_event", "symbol", "adj_close"])
)
adj_px

ts_event,symbol,adj_close
"datetime[ns, UTC]",cat,f64
2023-06-01 00:00:00 UTC,"""A""",116.0
2023-06-02 00:00:00 UTC,"""A""",118.5
2023-06-05 00:00:00 UTC,"""A""",118.88
2023-06-06 00:00:00 UTC,"""A""",116.83
2023-06-07 00:00:00 UTC,"""A""",116.0
…,…,…
2025-05-23 00:00:00 UTC,"""ZWS""",36.244262
2025-05-27 00:00:00 UTC,"""ZWS""",37.662205
2025-05-28 00:00:00 UTC,"""ZWS""",36.958334


In [5]:
daily_ret = (
    adj_px
    .with_columns(
        pl.col("adj_close").pct_change().over("symbol").alias("ret")
    )
    .pivot(
        on="symbol",
        on_columns=UNIVERSE,
        index="ts_event",
        values="ret"
    )
    .with_columns(
        pl.all().exclude("ts_event").fill_null(pl.all().exclude("ts_event").mean())
    )
)
daily_ret

ts_event,NVDA,SPY,TSLA,QQQ,AAPL,MSFT,AMZN,META,GOOGL,AVGO,AMD,TLT,GOOG,IWM,NFLX,TQQQ,MU,COST,ADBE,QCOM,COIN,SMCI,INTC,SMH,MSTR,TXN,AMAT,PEP,CSCO,BKNG,ASML,LRCX,TMUS,HYG,INTU,PDD,…,MMS,IGF,BITI,TNL,BANC,ASGN,OXM,ENOV,VSH,RLI,IDEV,BASE,LQDA,SCHR,VOE,ESNT,CHCO,PPH,LCII,AX,FNB,ESGD,SKYY,RITM,SHYG,TREE,AG,QQQE,ATGE,BKH,SPHY,FTSM,EFSC,TGNA,FIZZ,MAC,ANGL
"datetime[ns, UTC]",f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
2023-06-01 00:00:00 UTC,0.003019,0.000774,0.001821,0.000894,0.00039,0.000786,0.00123,0.002056,0.000857,0.00279,0.000383,-0.000165,0.00086,0.000459,0.002425,0.002203,0.001259,0.001584,0.000161,0.000808,0.004073,0.003518,-0.000324,0.001276,0.006855,0.000402,0.000683,-0.000478,0.000696,0.001712,0.000397,0.00092,0.001266,0.000375,0.00134,0.00129,…,-0.000028,0.000673,-0.002388,0.000922,0.000928,-0.000246,-0.000656,-0.000827,-0.000774,0.000609,0.000612,0.00038,0.002127,0.000082,0.000591,0.000678,0.000827,0.000434,0.000025,0.001523,0.000763,0.00059,0.001031,0.001066,0.000355,0.002482,0.00076,0.000591,0.002625,0.000209,0.000389,0.000203,0.000834,0.000293,0.000132,0.001616,0.000357
2023-06-02 00:00:00 UTC,-0.018389,0.014265,0.028723,0.007153,0.00288,0.005732,0.007795,-0.002565,0.008583,0.046124,-0.019866,-0.01222,0.008625,0.03682,-0.008794,0.021058,0.002756,0.007854,0.023196,-0.002845,0.018196,-0.038795,0.008021,-0.002457,0.006667,-0.002902,0.001038,0.007787,0.003814,0.025804,-0.001348,-0.012256,-0.062415,0.004987,0.020411,0.004909,…,0.018358,0.013362,-0.017066,0.049401,0.063182,0.043688,0.036568,0.045552,0.02292,0.0328,0.014316,0.031707,0.026559,-0.006997,0.025295,0.019651,0.058696,0.037384,0.065593,0.05783,0.048583,0.014016,0.020714,0.015663,-0.007319,0.042965,-0.016611,0.012101,0.038995,0.010083,0.004437,-0.006054,0.0483,0.029728,0.014582,0.08746,0.007356
2023-06-05 00:00:00 UTC,-0.00599,-0.003224,0.010242,-0.001381,-0.010933,0.001074,0.007493,-0.006245,0.010597,-0.011214,0.001959,0.000785,0.011029,-0.015533,0.005873,-0.003799,-0.022129,0.010566,-0.005839,-0.0268,-0.094328,-0.000897,-0.054106,-0.013617,-0.07947,-0.032067,-0.014005,0.004462,0.003399,0.020864,-0.007165,-0.013665,0.009674,-0.000134,0.025873,-0.022989,…,-0.024276,-0.004892,0.053496,0.001142,-0.020094,-0.020208,-0.001403,-0.015115,-0.025419,-0.013845,-0.005793,0.061466,-0.012373,0.001007,-0.003103,-0.025552,-0.030085,-0.020756,-0.020029,-0.031688,-0.020592,-0.005748,0.004139,0.027284,0.010568,0.031411,-0.006757,0.000672,-0.049275,0.014728,-0.000442,0.006768,-0.030324,-0.004914,-0.02004,-0.005329,-0.010588
2023-06-06 00:00:00 UTC,-0.005385,0.004313,0.041711,0.002342,-0.000447,-0.006617,0.011356,0.003586,0.010407,-0.019076,0.059924,0.007064,0.010355,0.029103,-0.006783,0.006538,0.002071,-0.001759,-0.002442,0.032691,-0.094887,0.061538,0.040713,0.006798,0.07964,0.002181,0.003833,-0.015656,-0.007373,0.013155,-0.005981,0.009584,-0.022281,0.000939,0.011694,0.073529,…,0.017613,0.005129,-0.055234,0.036248,0.077661,0.035651,0.05841,-0.00009,0.022218,0.024669,0.008534,-0.169265,-0.02164,0.0,0.009869,0.029556,0.050072,0.0,0.057087,0.050212,0.034604,0.009222,0.02021,0.012702,0.001216,0.071393,0.0,0.002551,0.020344,0.01403,0.001326,0.0,0.047273,-0.009259,-0.008469,0.05358,0.012177
2023-06-07 00:00:00 UTC,-0.026156,-0.004341,-0.008755,-0.01802,-0.006367,-0.032047,-0.040013,-0.032158,-0.036874,0.006156,-0.05413,-0.015782,-0.038179,0.018846,-0.00005,-0.054127,-0.005609,-0.010455,-0.034104,-0.009892,0.006066,0.014799,0.009053,-0.004548,-0.033784,0.012705,0.009957,-0.009356,-0.000402,-0.02718,-0.008795,0.000426,-0.013378,-0.004154,-0.05538,-0.011781,…,0.018761,0.004207,0.030646,-0.006115,0.046964,0.046942,-0.039067,0.031419,0.027783,0.021056,-0.009227,-0.131367,-0.012224,-0.004827,0.013531,0.010576,0.032493,-0.005202,0.020835,0.011893,0.02

In [6]:
causal_vol = daily_ret.with_columns(pl.all().exclude("ts_event").pow(2).ewm_mean(alpha=0.06, adjust=True).sqrt().shift(1))
st_ret = daily_ret.select(pl.all().exclude("ts_event")) / causal_vol.select(pl.all().exclude("ts_event"))
st_ret

NVDA,SPY,TSLA,QQQ,AAPL,MSFT,AMZN,META,GOOGL,AVGO,AMD,TLT,GOOG,IWM,NFLX,TQQQ,MU,COST,ADBE,QCOM,COIN,SMCI,INTC,SMH,MSTR,TXN,AMAT,PEP,CSCO,BKNG,ASML,LRCX,TMUS,HYG,INTU,PDD,PANW,…,MMS,IGF,BITI,TNL,BANC,ASGN,OXM,ENOV,VSH,RLI,IDEV,BASE,LQDA,SCHR,VOE,ESNT,CHCO,PPH,LCII,AX,FNB,ESGD,SKYY,RITM,SHYG,TREE,AG,QQQE,ATGE,BKH,SPHY,FTSM,EFSC,TGNA,FIZZ,MAC,ANGL
f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,…,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,…,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
-6.091545,18.432751,15.772806,7.997147,7.391423,7.294007,6.33717,-1.247801,10.011828,16.532114,-51.90181,-74.238497,10.033477,80.211513,-3.626217,9.55764,2.189277,4.958503,143.883853,-3.520703,4.466989,-11.025951,24.78048,-1.925194,0.972568,-7.217376,1.520629,16.281277,5.47842,15.068358,-3.394251,-13.328958,-49.288447,13.292265,15.226284,3.806144,40.005216,…,652.378678,19.846034,-7.146849,53.572913,68.048426,177.652462,55.70172,55.076899,29.62746,53.823373,23.378122,83.34176,12.486805,-85.471378,42.765199,28.965311,70.992474,86.154394,2587.03651,37.976968,63.663108,23.738934,20.086613,14.696575,-20.593977,17.307273,-21.845356,20.471423,14.854266,48.230407,11.394141,-29.784342,57.903712,101.498437,110.585282,54.115817,20.584887
-0.448033,-0.314363,0.495739,-0.266952,-5.243358,0.258752,1.323476,-2.677951,1.711711,-0.33805,0.137306,0.089521,1.772753,-0.587554,0.898634,-0.250001,-10.227178,1.838994,-0.35062,-12.650656,-7.056095,-0.032077,-9.38876,-6.893463,-11.758676,-15.256164,-15.838388,0.796689,1.222493,1.123885,-7.115947,-1.548897,0.215853,-0.037358,1.762036,-6.320674,-0.24387,…,-1.841875,-0.509278,4.326333,0.032193,-0.442927,-0.644249,-0.053435,-0.462105,-1.54388,-0.587843,-0.563075,2.699883,-0.646959,0.200366,-0.170807,-1.810081,-0.713834,-0.773277,-0.425312,-0.762968,-0.59029,-0.570719,0.277977,2.420989,2.008896,1.016695,-0.565989,0.077227,-1.756266,2.034084,-0.138169,1.556201,-0.874356,-0.230224,-1.914141,-0.08486,-2.002649
-0.475697,0.509624,2.358264,0.552748,-0.066474,-1.947941,1.782508,0.861483,1.295514,-0.694162,5.200026,0.999561,1.255059,1.25609,-1.075138,0.526377,0.155885,-0.225439,-0.17657,2.0381,-1.660182,2.737859,1.251552,0.823288,1.672957,0.113848,0.458208,-2.995865,-2.446149,0.677735,-1.37821,0.888298,-0.610846,0.325322,0.602819,5.256051,-0.291669,…,0.98321,0.621768,-1.656141,1.271154,2.02404,1.276483,2.765824,-0.003249,1.10552,1.194976,0.95278,-4.138575,-1.269736,0.0,0.670666,1.557851,1.306993,0.0,1.438737,1.30977,1.13098,1.049548,1.654333,0.683208,0.160463,2.29716,0.0,0.364326,0.54998,1.333628,0.513334,0.0,1.423698,-0.532116,-0.580266,1.059507,1.602339
-2.602669,-0.574426,-0.330113,-4.7248,-1.111047,-7.101304,-4.971573,-8.013542,-4.215715,0.241806,-1.647795,-2.233505,-4.301271,0.755823,-0.007766,-4.865798,-0.493187,-1.557607,-2.877047,-0.451845,0.087229,0.395064,0.258941,-0.576728,-0.580958,0.776255,1.344562,-1.00356,-0.08664,-1.517011,-1.815514,0.040703,-0.402944,-1.656185,-3.141515,-0.292543,-1.2759,…,1.052076,0.558999,0.756109,-0.19839,0.90051,1.552356,-1.101635,1.32692,1.342226,0.965017,-1.043341,-1.380483,-0.663765,-1.387607,0.997715,0.472757,0.776244,-0.245528,0.461816,0.283702,0.906942,-1.127977,-3.144367,0.862727,-0.598865,1.251813,-1.918303,-1.752079,0.242546,1.729394,-2.294881,-0.036979,0.523557,0.678927,0.410209,0.908242,0.280913
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
-0.318768,-0.400195,-0.144

In [7]:
X = st_ret.drop_nulls().to_numpy()

In [8]:
# Best rank k approximation of the returns covariance matrix
U, s, Vt = np.linalg.svd(X)
k = 10
U_k, S_k, Vt_k = U[:, :k], np.diag(s[:k]), Vt[:k, :]
F = U_k
L = S_k @ Vt_k

# Residuals: what the market factor can't explain
eps = X - F @ L          # (T, p)
D = np.diag(np.var(eps, axis=0))         # (2000, 2000) diagonal

# Factor covariance
Sigma_f = np.cov(F, rowvar=False)       # (10, 10)

# Model-implied return covariance: Σ_r = L^T Σ_f L + D
Sigma_r = L.T @ Sigma_f @ L + D         # (2000, 2000)


In [9]:
evals = np.linalg.eigvalsh(Sigma_r)

In [10]:
cond_n = np.abs(np.max(evals)) / np.abs(np.min(evals))
print(f"Condition Number: {cond_n}")

Condition Number: 24385159.967821464


In [13]:
# mkt_loadings = L[0,:] / np.mean(L[0,:])
np.save("loadings.npy", L)
np.save("factor_returns.npy", F)
np.save("idiosyncratic.npy", D)
np.save("factor_cov_matrix.npy", Sigma_f)
np.save("cov_matrix.npy", Sigma_r)


### Modeling eps (idiosyncratic) with GARCH(1,1)